## 1. Import Library & Load Dataset
Pada tahap ini, kita mengimpor library utama yang dibutuhkan seperti `pandas`, `numpy`, serta modul preprocessing dari `scikit-learn` (`MinMaxScaler`, `OneHotEncoder`, dan `OrdinalEncoder`). Selanjutnya, dataset COVID-19 dimuat dan difilter khusus untuk tingkat Provinsi.

In [35]:
import gdown
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder

file_id = '13N3wX7ULgh-uYw6GAMWm0eVq59GwcvNo'
url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, 'Dataset Covid19.csv', quiet=False)

df = pd.read_csv('Dataset Covid19.csv')

df_prov = df[df["Location Level"] == "Province"].copy()
df_wilayah = (
    df_prov.sort_values("Date").groupby("Location").last().reset_index()
)

df_wilayah.head()

Downloading...
From: https://drive.google.com/uc?id=13N3wX7ULgh-uYw6GAMWm0eVq59GwcvNo
To: /content/Dataset Covid19.csv
100%|██████████| 7.50M/7.50M [00:00<00:00, 39.3MB/s]


,Location,Date,Location ISO Code,New Cases,New Deaths,New Recovered,New Active Cases,Total Cases,Total Deaths,Total Recovered,...,Latitude,New Cases per Million,Total Cases per Million,New Deaths per Million,Total Deaths per Million,Total Deaths per 100rb,Case Fatality Rate,Case Recovered Rate,Growth Factor of New Cases,Growth Factor of New Deaths
0,Aceh,9/9/2022,ID-AC,10,0,13,-3,43999,2223,41681,...,4.225615,1.91,8385.14,0.00,423.65,42.36,5.05%,94.73%,0.71,1.0
1,Bali,9/9/2022,ID-BA,36,1,103,-68,166718,4726,161440,...,-8.369472,8.54,39542.51,0.24,1120.92,112.09,2.83%,96.83%,0.67,0.8
2,Banten,9/9/2022,ID-BT,316,0,1603,-1287,333014,2949,327558,...,-6.456736,29.47,31057.86,0.00,275.03,27.50,0.89%,98.36%,0.95,1.0
3,Bengkulu,9/9/2022,ID-BE,1,0,0,1,29149,522,28606,...,-3.533584,0.50,14577.86,0.00,261.06,26.11,1.79%,98.14%,0.33,1.0
4,DKI Jakarta,9/9/2022,ID-JK,1166,2,2146,-982,1408154,15505,1381349,...,-6.204699,107.50,129829.91,0.18,1429.54,142.95,1.10%,98.10%,0.88,2.0


## 2. Data Cleaning

### A. Handling Missing Values
Memeriksa keberadaan nilai yang hilang (null/missing values) pada setiap kolom, kemudian melakukan imputasi:
- Kolom Numerik: Diisi menggunakan nilai median.
- Kolom Kategorikal: Diisi menggunakan nilai modus.

In [41]:
print("=== Missing Values Sebelum Cleaning ===")
missing_info = df_wilayah.isnull().sum()
print(missing_info[missing_info > 0])

num_cols = df_wilayah.select_dtypes(include=["number"]).columns
for col in num_cols:
  if df_wilayah[col].isnull().sum() > 0:
    df_wilayah[col] = df_wilayah[col].fillna(df_wilayah[col].median())

cat_cols = df_wilayah.select_dtypes(exclude=["number"]).columns
for col in cat_cols:
  if df_wilayah[col].isnull().sum() > 0:
    df_wilayah[col] = df_wilayah[col].fillna(df_wilayah[col].mode()[0])

print(
    "\nJumlah total missing values setelah cleaning:",
    df_wilayah.isnull().sum().sum(),
)

=== Missing Values Sebelum Cleaning ===
City or Regency    34
dtype: int64

Jumlah total missing values setelah cleaning: 34


### B. Handling Duplicate Values & Outliers
1. Memeriksa dan menghapus baris data duplikat jika ada.
2. Penanganan outlier pada kolom numerik (`Total Cases`, `Total Deaths`, `Total Recovered`) menggunakan metode Interquartile Range (IQR) dengan pendekatan Capping / Winsorizing.

In [37]:
duplicate_count = df_wilayah.duplicated().sum()
print("Jumlah Data Duplikat:", duplicate_count)

if duplicate_count > 0:
  df_wilayah.drop_duplicates(inplace=True)
  df_wilayah.reset_index(drop=True, inplace=True)

cols_outlier = ["Total Cases", "Total Deaths", "Total Recovered"]

for col in cols_outlier:
  df_wilayah[col] = pd.to_numeric(df_wilayah[col], errors="coerce")

  Q1 = df_wilayah[col].quantile(0.25)
  Q3 = df_wilayah[col].quantile(0.75)
  IQR = Q3 - Q1

  lower_bound = Q1 - (1.5 * IQR)
  upper_bound = Q3 + (1.5 * IQR)

  df_wilayah[col] = df_wilayah[col].clip(
      lower=lower_bound, upper=upper_bound
  )

print("Penanganan data duplikat dan outlier selesai.")

Jumlah Data Duplikat: 0
Penanganan data duplikat dan outlier selesai.


## 3. Normalisasi Data Numerik
Melakukan normalisasi skala atribut numerik (`Total Cases`, `Total Deaths`, `Total Recovered`, `Population`) ke rentang 0 hingga 1 menggunakan MinMaxScaler.

In [38]:
scaler = MinMaxScaler()
numeric_cols = ["Total Cases", "Total Deaths", "Total Recovered", "Population"]

df_wilayah[[c + "_scaled" for c in numeric_cols]] = scaler.fit_transform(
    df_wilayah[numeric_cols]
)

df_wilayah[[c + "_scaled" for c in numeric_cols]].head()

,Total Cases_scaled,Total Deaths_scaled,Total Recovered_scaled,Population_scaled
0,0.094206,0.230594,0.090798,0.103315
1,0.478942,0.529961,0.475743,0.080151
2,1.000000,0.317426,1.000000,0.226316
3,0.047650,0.027150,0.048771,0.030354
4,1.000000,1.000000,1.000000,0.229096


## 4. Feature Encoding
Mengubah variabel/kolom kategorikal menjadi bentuk numerik:
- One-Hot Encoding: Diterapkan pada kolom nominal `Island` (tidak memiliki tingkat/urutan).
- Ordinal Encoding: Diterapkan pada kolom `Time Zone` (memiliki urutan alami: UTC+07:00 < UTC+08:00 < UTC+09:00).

In [39]:
ohe = OneHotEncoder(sparse_output=False)
island_encoded = ohe.fit_transform(df_wilayah[["Island"]])
island_df = pd.DataFrame(
    island_encoded, columns=ohe.get_feature_names_out(["Island"])
)
df_wilayah = pd.concat([df_wilayah, island_df], axis=1)

ordinal_encoder = OrdinalEncoder(
    categories=[["UTC+07:00", "UTC+08:00", "UTC+09:00"]]
)
df_wilayah["Time Zone_encoded"] = ordinal_encoder.fit_transform(
    df_wilayah[["Time Zone"]]
)

df_wilayah[["Time Zone", "Time Zone_encoded"] + list(island_df.columns)].head()

,Time Zone,Time Zone_encoded,Island_Jawa,Island_Kalimantan,Island_Maluku,Island_Nusa Tenggara,Island_Papua,Island_Sulawesi,Island_Sumatera
0,UTC+07:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,UTC+08:00,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,UTC+07:00,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,UTC+07:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,UTC+07:00,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Feature Engineering
Membuat fitur/variabel baru berdasarkan kombinasi atribut numerik yang ada untuk memberikan informasi analisis tambahan:
1. `Mortalitas_per_100rb_Penduduk`: Tingkat kematian per 100.000 penduduk.
2. `Kepadatan_Kasus`: Rasio total kasus terhadap luas wilayah (km²).

In [40]:
df_wilayah["Mortalitas_per_100rb_Penduduk"] = (
    df_wilayah["Total Deaths"] / df_wilayah["Population"]
) * 100000
df_wilayah["Kepadatan_Kasus"] = (
    df_wilayah["Total Cases"] / df_wilayah["Area (km2)"]
)

df_wilayah[[
    "Location",
    "Mortalitas_per_100rb_Penduduk",
    "Kepadatan_Kasus",
]].head()

,Location,Mortalitas_per_100rb_Penduduk,Kepadatan_Kasus
0,Aceh,42.364992,0.759179
1,Bali,112.092228,28.843945
2,Banten,27.503238,34.453068
3,Bengkulu,26.106017,1.463377
4,DKI Jakarta,79.807157,501.385542
